# Прогноз цветения: чистая версия исследования

Шершнёвское водохранилище. Задача — на дату T оценить вероятность
цветения на пригодной спутниковой сцене через 5–9 дней. Обучающая
метка получена из CV-классификатора пикселей; биологической
лабораторной метки здесь нет.

Порядок: источники и метка → временное разделение → сравнение
моделей → важность признаков → отдельный ежедневный график 2026 года.
Итоговая модель использует исходные спутниковые и погодные признаки,
сезон и спутник. Поля CV служат для построения обучающего ответа.

Ниже все сравниваемые методы используют одинаковые
хронологические фолды. Эта временная граница относится к
прогнозной модели. Предварительная пиксельная CV-модель и
порог для её меток создавались с участием ручных сцен
разных лет, включая 2026-й, поэтому сквозной независимой
проверки всей двухэтапной системы здесь нет.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import dates as mdates
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
LABEL_FILE = ROOT / "shershnevskoe_satellite_weather_cv_labeled.csv"
BASE_FILE = ROOT / "shershnevskoe_satellite_weather_merged.csv"
WEATHER_FILE = ROOT / "shershnevskoe_daily_weather_era5land.csv"
assert all(path.exists() for path in (LABEL_FILE, BASE_FILE, WEATHER_FILE)), (
    "Запустите ноутбук из папки hakaton: один из исходных CSV не найден."
)
plt.rcParams["figure.dpi"] = 115

## 1. Что находится в таблицах

В объединённой таблице одна строка соответствует дате и спутнику.
После CV-классификации к ней добавлены оценки текущей сцены и
метка будущего наблюдения. Погодный файл содержит ежедневные
данные ERA5-Land. Здесь используем готовую размеченную таблицу
из ноутбука 03 и проверяем ключевые правила её создания.

In [ ]:
labeled = pd.read_csv(
    LABEL_FILE, parse_dates=["date", "bloom_target_date_7d_050"]
).sort_values(["date", "sensor"]).reset_index(drop=True)
base = pd.read_csv(BASE_FILE, parse_dates=["date"])
weather = pd.read_csv(WEATHER_FILE, parse_dates=["date"])

source_summary = pd.DataFrame([
    {"источник": "Спутник + погода + CV", "строк": len(labeled),
     "начало": labeled.date.min().date(), "конец": labeled.date.max().date()},
    {"источник": "Спутник + погода", "строк": len(base),
     "начало": base.date.min().date(), "конец": base.date.max().date()},
    {"источник": "Ежедневная погода", "строк": len(weather),
     "начало": weather.date.min().date(), "конец": weather.date.max().date()},
])
display(source_summary)
assert not labeled.duplicated(["date", "sensor"]).any()

## 2. Что именно предсказываем

В старом ноутбуке пригодной считается CV-сцена с классифицированной
водой, долей артефактов не более 0,50 и известной средней CV-оценкой.
Средняя CV-оценка не менее 0,45 даёт текущую метку 1. Для даты T
выбирается ближайшая к T+7 пригодная сцена в окне T+5…T+9;
её метка становится ответом. Пустая метка означает отсутствие
подходящей будущей сцены, а не отсутствие цветения.

0,50 здесь относится к качеству сцены. Вероятность на выходе
прогнозной модели порогом 0,50 не обрезаем.

In [ ]:
TARGET = "bloom_in_7d_cv_050"
FUTURE_DATE = "bloom_target_date_7d_050"
labeled_rows = labeled.dropna(subset=[TARGET, FUTURE_DATE]).copy()
labeled_rows[TARGET] = labeled_rows[TARGET].astype(int)
labeled_rows["target_gap_days"] = (
    labeled_rows[FUTURE_DATE] - labeled_rows["date"]
).dt.days
assert labeled_rows["target_gap_days"].between(5, 9).all()
assert (labeled_rows["cv_artifact_fraction"] <= 0.50).all()
assert labeled_rows["cv_classified_water_area_m2"].gt(0).all()
assert labeled_rows["cv_mean_bloom_probability"].notna().all()
assert labeled_rows["bloom_now_cv_050"].eq(
    labeled_rows["cv_mean_bloom_probability"].ge(0.45).astype(int)
).all()

year_counts = labeled_rows.groupby(labeled_rows.date.dt.year).agg(
    примеров=(TARGET, "size"), цветений=(TARGET, "sum")
)
display(year_counts)
print(
    f"Всего {len(labeled_rows)} обучающих/проверочных строк; "
    f"{int(labeled_rows[TARGET].sum())} положительных меток."
)
current_2026 = labeled.loc[
    labeled.date.dt.year.eq(2026) & labeled["bloom_now_cv_050"].notna()
]
print(
    f"Текущих пригодных CV-сцен в 2026 году: {len(current_2026)}; "
    f"с цветением: {int(current_2026['bloom_now_cv_050'].sum())}. "
    "Это другая выборка, чем пары с будущей сценой."
)
display(
    current_2026.loc[current_2026["bloom_now_cv_050"].eq(1),
                     ["date", "sensor", "cv_mean_bloom_probability"]]
)

## 3. Признаки и порядок времени

На дату T модель получает индексы и качество спутникового снимка,
погоду за T, показатели погоды за предыдущие 7 и 14 дней,
сезон и тип спутника. Погодные окна в загрузочном скрипте
рассчитаны через сдвиг на один день, поэтому не берут значения
позже T. Погода дня T допустима при выпуске прогноза после
завершения этого дня.

CV-поля отсутствуют среди входных признаков HistGradient.
Спутниковые FAI разных сенсоров не сравниваются с одним
ручным порогом: спутник передаётся отдельным признаком.

In [ ]:
SAT_FEATURES = [
    "FAI_mean", "FAI_median", "FAI_std", "FAI_p90",
    "MNDWI_mean", "MNDWI_std", "MNDWI_p90",
    "Turbidity_FNU_mean", "Turbidity_FNU_std", "Turbidity_FNU_p90",
    "SPM_mgL_mean", "SPM_mgL_std", "SPM_mgL_p90",
    "B10_C_mean", "scene_cloud_pct", "clear_water_frac",
]
WEATHER_FEATURES = [
    "air_temp_c", "wind_speed_10m_ms", "precipitation_mm",
    "solar_radiation_mj_m2",
    "air_temp_mean_prev_7d", "lake_temp_mean_prev_7d",
    "precipitation_sum_prev_7d", "wind_mean_prev_7d",
    "calm_days_prev_7d", "solar_sum_prev_7d",
    "air_temp_mean_prev_14d", "lake_temp_mean_prev_14d",
    "precipitation_sum_prev_14d", "wind_mean_prev_14d",
]
NUMERIC_FEATURES = SAT_FEATURES + WEATHER_FEATURES + ["season_sin", "season_cos"]
INPUT_FEATURES = NUMERIC_FEATURES + ["sensor"]
assert not any(name.startswith(("cv_", "bloom_")) for name in INPUT_FEATURES)
assert set(INPUT_FEATURES).issubset(labeled_rows.columns)

# Без измерений на дату T медианная подстановка создавала бы фиктивный
# прогноз. В 2026 году одна размеченная строка не имеет дневной погоды.
model_rows = labeled_rows.loc[
    labeled_rows["FAI_mean"].notna() & labeled_rows["air_temp_c"].notna()
].copy()
model_rows["season_sin"] = np.sin(2 * np.pi * model_rows.date.dt.dayofyear / 365.25)
model_rows["season_cos"] = np.cos(2 * np.pi * model_rows.date.dt.dayofyear / 365.25)
print(
    f"Доступные входы и метка: {len(model_rows)} из {len(labeled_rows)} строк; "
    f"исключено {len(labeled_rows) - len(model_rows)}."
)
display(
    model_rows[NUMERIC_FEATURES].isna().mean().sort_values(ascending=False)
    .head(8).rename("доля пропусков")
)

## 4. Как обучается HistGradientBoosting

Это набор небольших деревьев решений. Признаки сначала делятся
на интервалы значений, затем деревья добавляются по очереди и
исправляют ошибки уже построенной модели. Параметры сохранены
из ноутбука 03: максимум 100 итераций, темп 0,05, до 8 листьев
в дереве, минимум 8 примеров в листе, L2-регуляризация 5.

Алгоритм воспринимает каждую строку как отдельный пример.
Порядок строк внутри fit не кодирует временную последовательность.
Время учитывают прошлые погодные окна, сезон и правило разделения:
для тестового года обучающие строки и даты их целевых сцен
должны находиться раньше 1 января тестового года.

In [ ]:
def make_pipeline(estimator):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("sensor", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["sensor"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", clone(estimator)),
    ])

ESTIMATORS = {
    "logistic_regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    ),
    "hist_gradient": HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.05, max_leaf_nodes=8,
        min_samples_leaf=8, l2_regularization=5.0, random_state=42
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=75, learning_rate=0.03, max_depth=2,
        min_samples_leaf=6, random_state=42
    ),
}

## 5. Расширяющаяся проверка по годам

Для 2022 года обучение идёт по более ранним годам, для 2023-го
добавляется 2022-й и так далее. Оценки моделей сравниваем по
2022–2025 годам. 2026 год остаётся отдельной проверкой и не
используется для выбора метода или его параметров.

Дополнительно проверяем дату будущей сцены для каждой обучающей
метки. Предобработка — заполнение пропусков и масштабирование —
обучается заново внутри каждого фолда.

In [ ]:
fold_rows = []
fold_predictions = []
historical_hist_fits = []
for test_year in (2022, 2023, 2024, 2025, 2026):
    cutoff = pd.Timestamp(f"{test_year}-01-01")
    train = model_rows.loc[
        (model_rows.date < cutoff) & (model_rows[FUTURE_DATE] < cutoff)
    ].copy()
    test = model_rows.loc[model_rows.date.dt.year.eq(test_year)].copy()
    if len(train) < 25 or train[TARGET].nunique() != 2 or test.empty:
        raise ValueError(f"Нельзя построить временной фолд {test_year}.")
    assert train[FUTURE_DATE].max() < test.date.min()
    assert not set(train[FUTURE_DATE]).intersection(set(test[FUTURE_DATE]))

    row = {
        "тестовый год": test_year, "train": len(train),
        "train: цветение": int(train[TARGET].sum()),
        "test": len(test), "test: цветение": int(test[TARGET].sum()),
        "последняя обучающая метка": train[FUTURE_DATE].max().date(),
        "первый прогноз": test.date.min().date(),
    }
    fold_rows.append(row)
    result = test[["date", "sensor", FUTURE_DATE, TARGET]].copy()
    result["test_year"] = test_year
    result["p_train_prevalence"] = train[TARGET].mean()
    for name, estimator in ESTIMATORS.items():
        fitted = make_pipeline(estimator)
        fitted.fit(train[INPUT_FEATURES], train[TARGET])
        result[f"p_{name}"] = fitted.predict_proba(test[INPUT_FEATURES])[:, 1]
        if name == "hist_gradient" and test_year < 2026:
            historical_hist_fits.append((test_year, fitted, test[INPUT_FEATURES].copy(), test[TARGET].copy()))
    fold_predictions.append(result)

folds = pd.DataFrame(fold_rows)
predictions = pd.concat(fold_predictions, ignore_index=True).sort_values(["date", "sensor"])
display(folds)
predictions.to_csv(ROOT / "04_walk_forward_predictions.csv", index=False, encoding="utf-8-sig")

## 6. Сравнение вероятностных моделей

Brier score — средняя квадратичная ошибка вероятности, меньше
лучше. ROC-AUC и PR-AUC показываем только как дополнительные
характеристики ранжирования; при малом числе положительных
ответов они нестабильны. Простая база — доля цветений в
обучающей части соответствующего года.

In [ ]:
def metric_row(name, frame):
    y = frame[TARGET].astype(int)
    p = frame[name].astype(float)
    return {
        "модель": name.removeprefix("p_"),
        "примеров": len(frame),
        "цветений": int(y.sum()),
        "Brier ↓": brier_score_loss(y, p),
        "ROC-AUC ↑": roc_auc_score(y, p) if y.nunique() == 2 else np.nan,
        "PR-AUC ↑": average_precision_score(y, p) if y.nunique() == 2 else np.nan,
        "средняя P": p.mean(),
    }

probability_columns = [c for c in predictions if c.startswith("p_")]
historical = predictions.loc[predictions.test_year.lt(2026)]
scores = pd.DataFrame([
    metric_row(column, historical) for column in probability_columns
]).sort_values("Brier ↓")
display(scores.round(3))
scores.to_csv(ROOT / "04_model_comparison_2022_2025.csv", index=False, encoding="utf-8-sig")

by_year = pd.DataFrame([
    {"год": year, **metric_row("p_hist_gradient", group)}
    for year, group in predictions.groupby("test_year")
])
display(by_year.round(3))
by_year.to_csv(ROOT / "04_hist_gradient_by_year.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
chart = scores.sort_values("Brier ↓", ascending=False)
ax.barh(chart["модель"], chart["Brier ↓"], color="#31708e")
ax.set(xlabel="Brier score: меньше лучше", title="Временная проверка 2022–2025")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

Во временной проверке 2022–2025 Random Forest имеет меньший Brier
score, чем HistGradientBoosting. Финальный график ниже оставлен
на HistGradientBoosting, чтобы воспроизвести прежний показательный
прогноз. Не следует объявлять HistGradient лучшим методом по
этому временному тесту.

## 7. Важность признаков HistGradientBoosting

У HistGradient нет встроенной таблицы важностей. Здесь каждый
признак перемешивается только внутри тестового года, после чего
измеряется изменение Brier score на исторических тестовых
фолдах 2022–2025. Большое положительное изменение означает,
что без исходного значения этого признака прогноз становится
хуже. Отрицательное возможно из-за шума и малой выборки.
Связанные признаки разделяют полезную информацию между собой;
этот график не показывает причинное влияние.

In [ ]:
rng = np.random.default_rng(42)
importance_rows = []
for feature in INPUT_FEATURES:
    weighted_change = 0.0
    total_rows = 0
    for year, fitted, X_test, y_test in historical_hist_fits:
        baseline_p = fitted.predict_proba(X_test)[:, 1]
        baseline_loss = brier_score_loss(y_test, baseline_p)
        changes = []
        for repeat in range(5):
            permuted = X_test.copy()
            values = permuted[feature].to_numpy(copy=True)
            rng.shuffle(values)
            permuted[feature] = values
            permuted_p = fitted.predict_proba(permuted)[:, 1]
            changes.append(brier_score_loss(y_test, permuted_p) - baseline_loss)
        weighted_change += np.mean(changes) * len(X_test)
        total_rows += len(X_test)
    importance_rows.append({
        "признак": feature, "ухудшение Brier при перемешивании": weighted_change / total_rows
    })
importance = pd.DataFrame(importance_rows).sort_values(
    "ухудшение Brier при перемешивании", ascending=False
).reset_index(drop=True)
display(importance.round(4))
importance.to_csv(ROOT / "04_hist_gradient_feature_importance.csv", index=False, encoding="utf-8-sig")

chart = importance.head(15).sort_values("ухудшение Brier при перемешивании")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(chart["признак"], chart["ухудшение Brier при перемешивании"], color="#3a7a72")
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Ухудшение Brier score", title="HistGradient: важность на будущих годах 2022–2025")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Отдельная проверка на 2026 году

В этом блоке вход — исходный спутниковый снимок и погода на
конкретную дату T. Метка описывает пригодную CV-сцену через
5–9 дней. 2026 год не участвовал в обучении этого фолда.
Это сравнение с предварительными CV-метками; число классов
проверяем перед вычислением ROC-AUC. В 2026 году есть четыре
положительные текущие CV-сцены, но ни одна доступная пара
«снимок сегодня → пригодный снимок через 5–9 дней» не получила
положительную будущую метку. Это ограничение разреженных пар,
а не доказательство отсутствия цветения.

In [ ]:
check_2026 = predictions.loc[predictions.test_year.eq(2026)].copy()
display(check_2026[["date", "sensor", FUTURE_DATE, TARGET, "p_hist_gradient"]])
display(pd.DataFrame([metric_row("p_hist_gradient", check_2026)]).round(3))
print(
    "Данные CV и итоговая модель не проверены независимыми пробами воды. "
    "Итоговый дневной график ниже использует между снимками последний пригодный снимок."
)

## 9. Ежедневный график 2026 года

Здесь HistGradientBoosting обучается заново на всех доступных
размеченных строках, у которых и дата входа, и дата будущей
CV-метки раньше 2026 года. Получается 72 обучающих примера.
На каждый день 2026 года берём погоду этого дня и последний
пригодный спутниковый снимок, если ему не более семи дней.

Это последовательность ежедневных прогнозов на следующие
5–9 дней, выпущенных в течение 2026 года. Год пока неполный:
ежедневная погода в файле есть только с мая по середину
сентября. На графике показывается только период доступных
прогнозов; разрывы внутри него означают отсутствие свежего
спутникового снимка.

In [ ]:
start_2026 = pd.Timestamp("2026-01-01")
final_train = model_rows.loc[
    (model_rows.date < start_2026) & (model_rows[FUTURE_DATE] < start_2026)
].copy()
assert len(final_train) == 72
final_model = make_pipeline(ESTIMATORS["hist_gradient"])
final_model.fit(final_train[INPUT_FEATURES], final_train[TARGET])
print(
    f"Финальное обучение: {len(final_train)} строк, "
    f"{int(final_train[TARGET].sum())} цветений, "
    f"{len(final_train) - int(final_train[TARGET].sum())} без цветения; "
    f"последняя дата метки {final_train[FUTURE_DATE].max().date()}."
)

daily_weather = weather.loc[weather.date.dt.year.eq(2026)].copy().sort_values("date")
daily_weather["season_sin"] = np.sin(2 * np.pi * daily_weather.date.dt.dayofyear / 365.25)
daily_weather["season_cos"] = np.cos(2 * np.pi * daily_weather.date.dt.dayofyear / 365.25)
image_rows = base.loc[
    base.date.dt.year.eq(2026)
    & base.usable_for_fai.fillna(False)
    & base.FAI_mean.notna(),
    ["date", "sensor"] + SAT_FEATURES,
].copy()
# На совпадающую дату сохраняем сцену с максимальной долей чистой воды.
image_rows = (
    image_rows.sort_values(["date", "clear_water_frac", "sensor"])
    .drop_duplicates("date", keep="last")
    .rename(columns={"date": "satellite_observation_date"})
)
daily_input = pd.merge_asof(
    daily_weather,
    image_rows.sort_values("satellite_observation_date"),
    left_on="date", right_on="satellite_observation_date",
    direction="backward",
)
daily_input["satellite_age_days"] = (
    daily_input.date - daily_input.satellite_observation_date
).dt.days
available = daily_input.satellite_age_days.between(0, 7)
daily_input["p_bloom_in_5_9_days"] = np.nan
daily_input.loc[available, "p_bloom_in_5_9_days"] = final_model.predict_proba(
    daily_input.loc[available, INPUT_FEATURES]
)[:, 1]

calendar = pd.DataFrame({"date": pd.date_range("2026-01-01", "2026-12-31")})
daily_output = calendar.merge(
    daily_input[[
        "date", "p_bloom_in_5_9_days", "satellite_observation_date",
        "satellite_age_days", "sensor",
    ]], on="date", how="left",
)
daily_output.to_csv(ROOT / "04_daily_probability_2026.csv", index=False, encoding="utf-8-sig")
print(
    f"Ежедневная погода: {len(daily_weather)} дней; "
    f"прогноз возможен для {int(daily_output.p_bloom_in_5_9_days.notna().sum())} дней. "
    f"Период прогноза: {daily_output.loc[daily_output.p_bloom_in_5_9_days.notna(), 'date'].min().date()} — "
    f"{daily_output.loc[daily_output.p_bloom_in_5_9_days.notna(), 'date'].max().date()}."
)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    daily_output.date, daily_output.p_bloom_in_5_9_days,
    color="#174c75", linewidth=2.0, label="Прогноз: P(цветение через 5–9 дней)"
)
new_image = daily_output.satellite_age_days.eq(0)
ax.scatter(
    daily_output.loc[new_image, "date"],
    daily_output.loc[new_image, "p_bloom_in_5_9_days"],
    color="#e6a23c", s=22, zorder=3, label="Новый пригодный снимок"
)
current_bloom = current_2026.loc[current_2026["bloom_now_cv_050"].eq(1)]
ax.scatter(
    current_bloom.date, np.ones(len(current_bloom)),
    marker="^", color="#288269", edgecolors="white", linewidths=0.5,
    s=78, zorder=5, label="CV: цветение на снимке в эту дату"
)
forecast_dates = daily_output.loc[daily_output.p_bloom_in_5_9_days.notna(), "date"]
ax.set(xlim=(forecast_dates.min() - pd.Timedelta(days=1),
             forecast_dates.max() + pd.Timedelta(days=1)),
       ylim=(-0.05, 1.05), xlabel="Дата выпуска прогноза в 2026 году",
       ylabel="Вероятность / CV-метка",
      )
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.grid(axis="y", alpha=0.25)
fig.suptitle("Шершнёвское водохранилище — ежедневный прогноз на 2026 год", y=0.99)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.94),
           ncol=3, fontsize=9, frameon=False)
fig.text(
    0.08, 0.035,
    "Разрывы: нет свежего пригодного снимка. Зелёные треугольники: "
    "CV отметила цветение на снимке в эту дату.",
    fontsize=9,
)
fig.subplots_adjust(top=0.82, bottom=0.19, left=0.08, right=0.98)
plt.savefig(ROOT / "04_daily_probability_2026.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Как читать результат

Синяя линия показывает, насколько программа считает вероятным цветение воды через примерно неделю после указанной даты. Это прогноз, а не результат измерения воды. Чем выше линия, тем больше оснований проверить водохранилище в ближайшие дни.
Почему почти всё лето линия высокая? Программа училась на небольшом числе прошлых наблюдений, и среди них цветение встречалось часто. Летняя погода и показатели спутниковых снимков 2026 года во многих случаях похожи на те условия, при которых программа раньше видела цветение. Кроме того, новый снимок есть не каждый день: между съёмками программа до семи дней использует данные последнего пригодного снимка.
Поэтому значение 0,9 не следует понимать как доказательство, что водоём цветёт сейчас или будет цвести девять дней из десяти. Оно означает: “по мнению нашей пока предварительной модели, риск высок”. Зелёные треугольники — отдельные даты, когда автоматический анализ снимка отметил цветение именно на снимке. Они помогают сопоставлять прогноз с наблюдениями, но тоже требуют проверки.
Практический смысл графика — подсказать, когда особенно полезно взять пробу воды или внимательно осмотреть водоём. Утверждать по одной синей линии, что водохранилище непрерывно цвело всё лето, нельзя